# E0 — La moneta: una PFN verificata contro la formula esatta

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emanbuc/corso_foundation_tabular/blob/main/notebooks/costruire-tfm/E0_moneta.ipynb)

**Corso 2 · Costruire un TFM — Modulo 1 (laboratorio, 20 min)**

Una rete addestrata su dati campionati da un prior impara ad approssimare la predittiva bayesiana di quel prior. Qui lo verifichiamo su un caso in cui la risposta esatta è nota.

- **Prior:** una moneta con probabilità di testa $p \sim \mathrm{Beta}(\alpha, \beta)$.
- **Contesto:** $n$ lanci della moneta.
- **Compito:** predire il lancio successivo.

Predittiva esatta, con $k$ teste su $n$ lanci:

$$p(y_{n+1} = 1 \mid y_{1:n}) = \frac{\alpha + k}{\alpha + \beta + n}$$

Il notebook gira anche su CPU.

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("Dispositivo:", device)

## 1. Il prior: un generatore di dataset

Ogni "dataset" è una sequenza di lanci di una moneta diversa. Per ogni riga del batch:

1. si estrae la probabilità di testa `p` dalla Beta;
2. si generano `n_max` lanci di contesto più un lancio in più, che è l'etichetta nascosta;
3. si estrae quanti lanci di contesto mostrare alla rete, `n` tra 0 e `n_max`.

In [ ]:
N_MAX = 20  # lunghezza massima del contesto vista in addestramento


def campiona_prior(batch, alpha, beta, n_max=N_MAX):
    p = torch.distributions.Beta(alpha, beta).sample((batch, 1)).to(device)
    lanci = torch.bernoulli(p.expand(batch, n_max + 1)).long()
    contesto, etichetta = lanci[:, :n_max], lanci[:, n_max].float()
    n = torch.randint(0, n_max + 1, (batch,), device=device)
    return contesto, n, etichetta


contesto, n, etichetta = campiona_prior(5, alpha=1.0, beta=1.0)
for c, m, e in zip(contesto.tolist(), n.tolist(), etichetta.tolist()):
    print(f"n = {m:2d}  contesto = {c[:m]}  →  lancio da predire: {int(e)}")

## 2. La soluzione esatta

Serve solo per il confronto finale: la rete non la vede mai.

In [ ]:
def predittiva_esatta(k, n, alpha, beta):
    return (alpha + k) / (alpha + beta + n)


print(predittiva_esatta(k=3, n=4, alpha=1.0, beta=1.0))  # 4 lanci, 3 teste → 4/6

## 3. Il modello: un transformer minuscolo

Ogni lancio è un token (`0` = croce, `1` = testa). In coda si aggiunge un token "domanda" (`2`): la sua uscita è la predizione per il lancio successivo.

Due scelte che ritroveremo nel modello tabulare del modulo 3:

- **nessun embedding posizionale**: l'ordine dei lanci non conta, quindi il modello non deve poterlo usare;
- **una maschera** nasconde i lanci oltre i primi `n`, così lo stesso batch contiene contesti di lunghezze diverse.

In [ ]:
class PFNMoneta(nn.Module):
    def __init__(self, d=32, teste=2, strati=2):
        super().__init__()
        self.embedding = nn.Embedding(3, d)  # 0 = croce, 1 = testa, 2 = domanda
        strato = nn.TransformerEncoderLayer(
            d, teste, dim_feedforward=4 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(strato, strati, enable_nested_tensor=False)
        self.uscita = nn.Linear(d, 1)

    def forward(self, contesto, n):
        batch, lunghezza = contesto.shape
        domanda = torch.full((batch, 1), 2, device=contesto.device)
        token = torch.cat([contesto, domanda], dim=1)
        # True = posizione da ignorare: i lanci oltre i primi n
        nascosti = torch.arange(lunghezza, device=contesto.device)[None, :] >= n[:, None]
        nascosti = torch.cat([nascosti, torch.zeros_like(nascosti[:, :1])], dim=1)
        h = self.encoder(self.embedding(token), src_key_padding_mask=nascosti)
        return self.uscita(h[:, -1]).squeeze(-1)  # logit di "testa" per il lancio successivo


modello = PFNMoneta()
print("Parametri:", sum(p.numel() for p in modello.parameters()))

## 4. L'addestramento

L'obiettivo è quello di ogni PFN: si campiona un dataset dal prior, si nasconde un'etichetta, si minimizza la cross-entropy sull'etichetta nascosta.

$$\mathcal{L}(\theta) = \mathbb{E}_{D \sim p(D)} \left[ -\log q_\theta(y_{\text{test}} \mid D_{\text{train}}) \right]$$

Ogni passo usa un batch di monete nuove: la rete non vede mai due volte gli stessi dati.

In [ ]:
def addestra(alpha, beta, passi=3000, batch=256, lr=1e-3):
    modello = PFNMoneta().to(device)
    ottimizzatore = torch.optim.Adam(modello.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ottimizzatore, passi)
    storia = []
    inizio = time.perf_counter()
    for passo in range(passi):
        contesto, n, etichetta = campiona_prior(batch, alpha, beta)
        loss = F.binary_cross_entropy_with_logits(modello(contesto, n), etichetta)
        ottimizzatore.zero_grad()
        loss.backward()
        ottimizzatore.step()
        scheduler.step()
        storia.append(loss.item())
        if (passo + 1) % 500 == 0:
            media = sum(storia[-500:]) / 500
            print(f"passo {passo + 1:5d}  loss media = {media:.4f}")
    print(f"Tempo di addestramento: {time.perf_counter() - inizio:.0f} s")
    return modello.eval(), storia


ALPHA, BETA = 1.0, 1.0
modello, storia = addestra(ALPHA, BETA)

### Quanto vale la loss nel suo minimo

Il minimo della loss è la distribuzione predittiva a posteriori. Possiamo stimare quanto vale la loss in quel minimo applicando la formula esatta a molte monete campionate dal prior, e confrontarla con la curva di addestramento.

La curva della rete oscilla per il rumore dei batch, quindi a tratti passa sotto la linea: in media, però, nessun modello può fare meglio della predittiva esatta.

In [ ]:
@torch.no_grad()
def loss_ottima(alpha, beta, campioni=200_000):
    contesto, n, etichetta = campiona_prior(campioni, alpha, beta)
    visibili = torch.arange(N_MAX, device=device)[None, :] < n[:, None]
    k = (contesto * visibili).sum(dim=1)
    p = predittiva_esatta(k, n, alpha, beta)
    return F.binary_cross_entropy(p, etichetta).item()


ottima = loss_ottima(ALPHA, BETA)
media_mobile = torch.tensor(storia).unfold(0, 100, 1).mean(dim=1)

plt.figure(figsize=(6, 3.5))
plt.plot(range(100, len(storia) + 1), media_mobile, label="rete (media mobile su 100 passi)")
plt.axhline(ottima, color="k", linestyle="--", label=f"predittiva esatta ({ottima:.4f})")
plt.xlabel("passo")
plt.ylabel("cross-entropy")
plt.legend()
plt.show()

## 5. Il confronto: rete contro formula esatta

Per ogni coppia $(n, k)$ con $0 \le k \le n \le$ `N_MAX` si costruisce una sequenza con $k$ teste e si confronta l'uscita della rete con la formula.

**Grafico da commentare:** i punti devono cadere sulla diagonale.

In [ ]:
@torch.no_grad()
def confronta(modello, alpha, beta, n_max=N_MAX):
    """Restituisce n, k, predizione della rete e valore esatto per tutte le coppie (n, k)."""
    coppie = [(n, k) for n in range(n_max + 1) for k in range(n + 1)]
    n = torch.tensor([c[0] for c in coppie], device=device)
    k = torch.tensor([c[1] for c in coppie], device=device)
    # k teste nelle prime posizioni: l'ordine non conta
    contesto = (torch.arange(n_max, device=device)[None, :] < k[:, None]).long()
    rete = torch.sigmoid(modello(contesto, n))
    esatta = predittiva_esatta(k, n, alpha, beta)
    return n.cpu(), k.cpu(), rete.cpu(), esatta.cpu()


def grafico_diagonale(modello, alpha, beta):
    n, k, rete, esatta = confronta(modello, alpha, beta)
    plt.figure(figsize=(4.5, 4.5))
    punti = plt.scatter(esatta, rete, c=n, s=14, cmap="viridis")
    plt.plot([0, 1], [0, 1], "k--", linewidth=1)
    plt.colorbar(punti, label="n (lanci di contesto)")
    plt.xlabel("formula esatta")
    plt.ylabel("uscita della rete")
    plt.title(f"Beta({alpha}, {beta})")
    plt.show()
    print(f"Errore assoluto massimo: {(rete - esatta).abs().max():.3f}")


grafico_diagonale(modello, ALPHA, BETA)

Con zero lanci la rete deve restituire la media del prior, $\alpha / (\alpha + \beta)$: non ha visto nessun dato, eppure la risposta è corretta perché l'ha imparata dal prior.

In [ ]:
n, k, rete, esatta = confronta(modello, ALPHA, BETA)
for riga in [0, 1, 2, 65, 230]:
    print(f"n = {int(n[riga]):2d}  k = {int(k[riga]):2d}  rete = {rete[riga]:.3f}  esatta = {esatta[riga]:.3f}")

## 6. Cosa generalizza e cosa no

In addestramento la rete ha visto al massimo `N_MAX` lanci. Il modello accetta sequenze più lunghe senza modifiche: cosa risponde?

La formula esatta vale per qualsiasi $n$. La rete invece ha imparato solo dove il prior le ha mostrato dati.

In [ ]:
n, k, rete, esatta = confronta(modello, ALPHA, BETA, n_max=100)
errore = (rete - esatta).abs()
errore_massimo = [errore[n == m].max().item() for m in range(101)]

plt.figure(figsize=(6, 3.5))
plt.plot(range(101), errore_massimo)
plt.axvline(N_MAX, color="k", linestyle="--", label="lunghezza massima vista in addestramento")
plt.xlabel("n (lanci di contesto)")
plt.ylabel("errore assoluto massimo")
plt.legend()
plt.show()

**Messaggio:** il modello è buono quanto il prior da cui ha imparato.

Domande per la discussione:

- Dove cresce l'errore, e perché proprio lì?
- Cosa cambierebbe addestrando con `N_MAX = 100`?
- Quale parte di questo notebook corrisponde al "pre-training" di un TFM, e quale all'"inferenza"?

## Esercizio

Riaddestrare con $\mathrm{Beta}(0.5, 0.5)$ al posto di $\mathrm{Beta}(1, 1)$ e verificare che la rete segua la nuova formula.

Stessa architettura, prior diverso, modello diverso.

Suggerimenti:

- `addestra` e `grafico_diagonale` accettano già `alpha` e `beta`;
- confrontare le due reti sullo stesso contesto, per esempio 2 lanci e 2 teste: quale delle due si fida di più dei dati?

In [ ]:
# TODO: addestrare una seconda rete con Beta(0.5, 0.5)
# modello_b, storia_b = ...

# TODO: disegnare il grafico della diagonale per la nuova rete

# TODO: confrontare le due reti su n = 2, k = 2